IN this notebook we will use CTAS and COPY into Commands to ingest data, you should have a catalog a schema and a volume. we are using kp as catalog, deafukt as schema and fordata as volume. you should create these before using the notebook


In [0]:

use catalog kp;
use schema default;
SELECT current_catalog(), current_schema()

In [0]:
%python
import os
os.listdir('/Volumes/kp/default/fordata')


In [0]:

SELECT * 
FROM read_files(
  '/Volumes/kp/default/fordata/userdata',
  format => 'parquet'
)
LIMIT 10;

In [0]:

-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS historical_users_bronze_ctas_rf;


-- Create the Delta table
CREATE TABLE historical_users_bronze_ctas_rf 
SELECT * 
FROM read_files(
        '/Volumes/kp/default/fordata/userdata',
        format => 'parquet'
      );


-- Preview the Delta table
SELECT * 
FROM historical_users_bronze_ctas_rf 
LIMIT 10;

In [0]:

DESCRIBE TABLE EXTENDED historical_users_bronze_ctas_rf

Incremental Data Ingestion with COPY INTO
COPY INTO is a Databricks SQL command that allows you to load data from a file location into a Delta table. This operation is re-triable and idempotent, i.e. files in the source location that have already been loaded are skipped. This command is useful for when you need to load data into an existing Delta table.

COPY INTO

Example 1: Common Schema Mismatch Error
The cell below creates an empty table named historical_users_bronze_ci with a defined schema for only the user_id and user_first_touch_timestamp columns.

However, the Parquet files in '/Volumes/dbacademy_ecommerce/v01/raw/users-historical' contain three columns:

user_id
user_first_touch_timestamp
email
Run the cell below and review the error. You should see the [COPY_INTO_SCHEMA_MISMATCH_WITH_TARGET_TABLE] error. This error occurs because there is a schema mismatch: the Parquet files contain 3 columns, but the target table historical_users_bronze_ci only has 2 columns.

How can you handle this error?

In [0]:


--This cell returns an error


-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS historical_users_bronze_ci;


--Create an empty table with the specified table schema (only 2 out of the 3 columns)
CREATE TABLE historical_users_bronze_ci (
  user_id STRING,
  user_first_touch_timestamp BIGINT
);


--Use COPY INTO to populate Delta table
COPY INTO historical_users_bronze_ci
  FROM '/Volumes/kp/default/fordata/userdata'
  FILEFORMAT = parquet;

We can fix this error by adding COPY_OPTIONS with the mergeSchema = 'true' option. When set to true, this option allows the schema to evolve based on the incoming data.

Run the next cell with the COPY_OPTIONS option added. You should notice that the Parquet files were successfully ingested into the table.

In [0]:
COPY INTO historical_users_bronze_ci
  FROM '/Volumes/kp/default/fordata/userdata'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');     -- Merge the schema of each file

In [0]:
SELECT *
FROM historical_users_bronze_ci
LIMIT 10;

Preemptively Handling Schema Evolution
Another way to ingest the same files into a Delta table is to start by creating an empty table named historical_users_bronze_ci_no_schema.

Then, add the COPY_OPTIONS ('mergeSchema' = 'true') option to enable schema evolution for the table.

Run the cell and confirm that  rows were added to the Delta table.

In [0]:
-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS historical_users_bronze_ci_no_schema;


-- Create an empty table without the specified schema
CREATE TABLE historical_users_bronze_ci_no_schema;


-- Use COPY INTO to populate Delta table
COPY INTO historical_users_bronze_ci_no_schema
  FROM '/Volumes/kp/default/fordata/userdata'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');

Idempotency (Incremental Ingestion)
COPY INTO tracks the files it has previously ingested. If the command is run again, no additional data is ingested because the files in the source directory haven't changed.

In [0]:
COPY INTO historical_users_bronze_ci_no_schema
  FROM '/Volumes/kp/default/fordata/userdata'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');